# TODO: 
Format Everything (mostly according to the project outline)

## Libraries & Packages

In [118]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## Data Loading

In [161]:
# Yearly appropriations for NASA compared to all other organizations
fed_budgets_raw = pd.read_csv("data/federal_budgets.csv")

# Yearly allocations to the constituent divisions of NASA's Science Directorate
division_budgets_raw = pd.read_csv("data/nasa_division_budgets.csv")

# Mean cont of sunspots per month
sunspots_raw = pd.read_json("data/sunspot_records.json")

# Date limits of each solar cycle
solar_cycle_raw = pd.DataFrame({
    'cycle_num' : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25],
    'start_date' : 	["1755-02", "1766-06", "1775-06", "1784-09", "1798-04", "1810-07",
                     "1823-05", "1833-11", "1843-07", "1855-12", "1867-03", "1878-12",
                     "1890-03", "1902-01", "1913-07", "1923-08", "1933-09", "1944-02",
                     "1954-04", "1964-10", "1976-03", "1986-09", "1996-08", "2008-12", "2019-12"],
    'end_date' : ["1766-06", "1775-06", "1784-09", "1798-04", "1810-07",
                  "1823-05", "1833-11", "1843-07", "1855-12", "1867-03", "1878-12",
                  "1890-03", "1902-01", "1913-07", "1923-08", "1933-09", "1944-02",
                  "1954-04", "1964-10", "1976-03", "1986-09", "1996-08", "2008-12", "2019-12", "2030-06"],
    'expected_start' : ["1755-02", "1766-02", "1777-02", "1788-02", "1799-02", "1810-02", "1821-02", "1832-02",
                        "1843-02", "1854-02", "1865-02", "1876-02", "1887-02", "1898-02", "1909-02", "1920-02",
                        "1931-02", "1942-02", "1953-02", "1964-02", "1975-02", "1986-02", "1997-02", "2008-02","2019-02"],
    'expected_end' : ["1766-02", "1777-02", "1788-02", "1799-02", "1810-02", "1821-02", "1832-02",
                      "1843-02", "1854-02", "1865-02", "1876-02", "1887-02", "1898-02", "1909-02", "1920-02",
                      "1931-02", "1942-02", "1953-02", "1964-02", "1975-02", "1986-02", "1997-02", "2008-02","2019-02", '2030-02']
})



## Cleaning & Transformations

### Transformation Helper Functions

In [ ]:
# Given a date, returns which observed solar cycle that date is in
def getObservedCycle(date, df) :
    
    # Check for which solar cycle a date falls in
    mask = (df['start_date'] <= date) & (date < df['end_date'])
    match = df[mask]
    
    if not match.empty:
        return match.iloc[0]['cycle_num'] 
    
    # Either the date is in the future, or it's before the first official solar cycle in 1755
    return np.nan


# Given a date, returns the expected solar cycle assuming a consistent 11-year cycle
def getExpectedCycle(date, df) :

    mask = (df['expected_start'] <= date) & (date < df['expected_end'])
    match = df[mask]
    
    if not match.empty:
        return match.iloc[0]['cycle_num'] 
    
    # Either the date is in the future, or it's before the first official solar cycle in 1755
    return np.nan


### Federal Budgets Dataset

In [121]:
print(fed_budgets_raw.dtypes)

fed_budgets_raw.head()

gov_fiscal_year               int64
president                    object
budget_nominal               object
budget_real                  object
budget_real_yearly_delta    float64
inflation_cuml              float64
departments_group            object
dtype: object


,gov_fiscal_year,president,budget_nominal,budget_real,budget_real_yearly_delta,inflation_cuml,departments_group
0,1960,Eisenhower,523,"5,547",0.000000,10.606554,nasa
1,1961,Eisenhower,964,"10,225",0.843212,10.606554,nasa
2,1962,Eisenhower,"1,825","19,152",0.873098,10.494190,nasa
3,1963,Kennedy,"3,674","38,099",0.989303,10.369879,nasa
4,1964,Kennedy,"5,100","52,239",0.371135,10.242901,nasa


In [122]:

fed_budgets = fed_budgets_raw.copy()

# Create datetime attributes
fed_budgets['gfy_start'] = pd.to_datetime(fed_budgets['gov_fiscal_year'], format='%Y') - pd.DateOffset(months=3)
fed_budgets['gfy_end'] = fed_budgets['gfy_start'] + pd.DateOffset(months=12)


### Science Divisions Budgets Dataset

In [123]:
division_budgets = division_budgets_raw.copy()

# Create datetime attributes
division_budgets['gfy_start'] = pd.to_datetime(division_budgets['gov_fiscal_year'], format='%Y') - pd.DateOffset(months=3)
division_budgets['gfy_end'] = division_budgets['gfy_start'] + pd.DateOffset(months=12)


In [124]:
fed_budgets.head()

,gov_fiscal_year,president,budget_nominal,budget_real,budget_real_yearly_delta,inflation_cuml,departments_group,gfy_start,gfy_end
0,1960,Eisenhower,523,"5,547",0.000000,10.606554,nasa,1959-10-01,1960-10-01
1,1961,Eisenhower,964,"10,225",0.843212,10.606554,nasa,1960-10-01,1961-10-01
2,1962,Eisenhower,"1,825","19,152",0.873098,10.494190,nasa,1961-10-01,1962-10-01
3,1963,Kennedy,"3,674","38,099",0.989303,10.369879,nasa,1962-10-01,1963-10-01
4,1964,Kennedy,"5,100","52,239",0.371135,10.242901,nasa,1963-10-01,1964-10-01


### Solar Cycle Dataset

In [125]:
solar_cycle = solar_cycle_raw.copy()

# Cast as datetime
solar_cycle['start_date'] = pd.to_datetime(solar_cycle['start_date'])
solar_cycle['end_date'] = pd.to_datetime(solar_cycle['end_date'])

### Sunspots Dataset

In [ ]:
sunspots = sunspots_raw.copy()

# Reformat columns
sunspots = sunspots.drop(labels=['smoothed_ssn', 'observed_swpc_ssn', 'smoothed_swpc_ssn', 'f10.7', 'smoothed_f10.7'], axis=1)
sunspots = sunspots.rename(columns={'time-tag' : 'date'})

# Cast as datetime
sunspots['date'] = pd.to_datetime(sunspots['date'])

# Set solar Observed and Expected cycle numbers
sunspots['cycle_obsv'] = sunspots['date'].apply(lambda x: getObservedCycle(x, solar_cycle)).astype('Int64')
sunspots['cycle_expc'] = sunspots['date'].apply(lambda x: getExpectedCycle(x, solar_cycle)).astype('Int64')


# Drop rows from before the first recorded solar cycle
sunspots = sunspots.dropna().reset_index(drop=True)

# Assign rows a government fiscal year 
# (runs from October of the preceding calendar year through September of the current calendar year)
# Example: December 2025 is in US Fiscal Year 2026
sunspots['gov_fiscal_year'] = sunspots['date'].dt.to_period('Y-SEP')

# An 11-year solar cycle is supposed to start and end in periods of low solar activity
# Assign 'high_period' and 'low_period' flags accordingly



sunspots_nasa = sunspots.loc[sunspots['date'].dt.year >= 1960].copy().reset_index(drop=True)

# Need to group into government fiscal years (Oct-Sept), 
# then categorize by position in the cycle

In [160]:
sunspots.groupby('cycle_num').size()

cycle_num
1     136
2     108
3     111
4     163
5     147
6     154
7     126
8     116
9     149
10    135
11    141
12    135
13    142
14    138
15    121
16    121
17    125
18    122
19    126
20    137
21    126
22    119
23    148
24    132
25     72
dtype: int64